you must insert this at the top of a provider to simular errors and generate an error log

if input.get("retry_count", 0) < 2:

    raise RuntimeError("🔥 Simulated failure to trigger retry logic")

In [1]:
from pathlib import Path
import shutil
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

# Define the directories to be emptied
directories_to_clear = [
    PROJECT_ROOT / "working_files",
    PROJECT_ROOT / "extensions",
    PROJECT_ROOT / "experiments" / "snapshots"
]

# Function to delete all files and directories within a specified directory
def clear_directory(directory: Path):
    if directory.exists() and directory.is_dir():
        for item in directory.iterdir():
            try:
                if item.is_dir():
                    shutil.rmtree(item)  # Remove directory and all its contents
                else:
                    item.unlink()  # Remove file
            except Exception as e:
                print(f"Error deleting {item}: {e}")

# Clear all the directories
for directory in directories_to_clear:
    clear_directory(directory)

print("All specified directories have been cleared.")

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


Working directory is now: C:\Repos\codecritic
All specified directories have been cleared.
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 0e312eae-43b1-4db0-9a0d-e26e0062ad44
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 6368e9df-ca12-4187-a2ab-a4a9438075c4
Seeded SystemPrompt 'format' with ID: 1 and GUID: 1faefea8-f9ca-4f4c-90aa-0886cc5b0295
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: cbbcff0a-f843-4ca2-a246-f892d4270c89
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
✅ Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
✅ Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE
from app.enums.system_enums import SYSTEM
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/complex_lint_task.py")

# Construct input using session metadata
input_data = {
    "file_path":  str(file),
    "session_id": session_row.id,
    "system":     SYSTEM.LINTING.value
}

# Run associated program
from app.enums.logging_enums import RunContext, PROVIDER_TYPE

context = RunContext(
    called_by_type=PROVIDER_TYPE.SESSION,
    called_by_id=session_row.id,
    session_id=session_row.id,
    file_log_id=None,  # if available, otherwise use None
    execution_chain=[]
)

program = ProgramProviderFactory.create(
    id=session_row.program_provider_id,
    context=context
)

result = program.run(input_data, context=context)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


✅ Final Program Result (via SessionConfig):
{
  "state": "end",
  "previous_state": "preprocessing_controller",
  "state_type": "end",
  "decision": "accepted",
  "steps": 2,
  "max_steps": 20,
  "summary": "completed successfully",
  "output": {
    "state": "end",
    "file_path": "working_files\\complex_lint_task.py",
    "session_id": 1,
    "file_log_id": 1,
    "reason": "completed successfully",
    "steps": 2,
    "retry_count": 0,
    "_last_state": "preprocessing_controller",
    "decision": "accepted",
    "original_file": "tests\\complex_lint_task.py",
    "run_id": "d8d7f030-a681-43df-883b-94445e0fa23b",
    "transition_metadata": {
      "original_file": "tests\\complex_lint_task.py",
      "run_id": "d8d7f030-a681-43df-883b-94445e0fa23b",
      "agent_output": {},
      "file_path": null
    },
    "state_output": {
      "state": "end",
      "previous_state": "linting",
      "state_type": "end",
      "decision": "accepted",
      "steps": 3,
      "max_steps": 10,
  

In [5]:
import sqlite3
import pandas as pd
from datetime import datetime
from app.db.connection import DB_PATH

# 🗂️ Load from SQLite
conn = sqlite3.connect(DB_PATH)
df_error = pd.read_sql("SELECT * FROM error_log WHERE session_id='1'", conn)
conn.close()

# 📅 Normalize timestamps and sort
df_error["timestamp"] = pd.to_datetime(df_error["timestamp"])
df_error = df_error.sort_values("timestamp").reset_index(drop=True)

# 🧠 Print structured error entries
for i, row in df_error.iterrows():
    print(f"\n🧩 Error Log {i + 1}")
    print(f"🕒 {row['timestamp']}")
    print(f"🧬 Run ID   : {row['run_id']}")
    print(f"📄 File Log : {row.get('file_log_id')}")
    print(f"🚨 Error Type: {row['error_type']}")
    print(f"📦 Provider : {row['provider_type']} (ID {row['provider_id']})")
    print(f"🧬 Called By: {row['called_by_type']} (ID {row['called_by_id']})")
    print(f"🧱 Parent ID: {row.get('parent_id')}")
    chain_raw = row.get("execution_chain", "[]")
    chain_list = json.loads(chain_raw) if isinstance(chain_raw, str) else chain_raw
    print(f"📍 Chain Len: {len(chain_list)}")
    print(f"⏱️ Latency  : {row.get('latency_ms')} ms")
    print("💬 Message  :")
    print(row["message"].strip())
    print("—" * 80)



🧩 Error Log 1
🕒 2025-06-17 11:07:32.655852+00:00
🧬 Run ID   : b015c6ab-cc4b-4ca5-9909-10c41b79190b
📄 File Log : 1
🚨 Error Type: runtime
📦 Provider : PROVIDER_TYPE.AGENT (ID 2)
🧬 Called By: PROVIDER_TYPE.SESSION (ID 1)
🧱 Parent ID: eaefb45b-c9cb-4eec-94fe-7f6a78d89a06
📍 Chain Len: 5
⏱️ Latency  : 0 ms
💬 Message  :
🔥 Simulated failure to trigger retry logic
————————————————————————————————————————————————————————————————————————————————

🧩 Error Log 2
🕒 2025-06-17 11:07:32.913849+00:00
🧬 Run ID   : a3137b1f-c0ef-42ff-b1b1-77e37ca22e0b
📄 File Log : 1
🚨 Error Type: runtime
📦 Provider : PROVIDER_TYPE.AGENT (ID 2)
🧬 Called By: PROVIDER_TYPE.SESSION (ID 1)
🧱 Parent ID: b015c6ab-cc4b-4ca5-9909-10c41b79190b
📍 Chain Len: 6
⏱️ Latency  : 0 ms
💬 Message  :
🔥 Simulated failure to trigger retry logic
————————————————————————————————————————————————————————————————————————————————


In [4]:
import sqlite3
import json
from pathlib import Path
import pandas as pd
from app.db.connection import DB_PATH

# 🔍 Step 1: Load provider logs
conn = sqlite3.connect(DB_PATH)
df_provider = pd.read_sql("SELECT * FROM provider_log", conn)

# 🔍 Step 2: Load error log (or use specific run ID if needed)
df_error = pd.read_sql("SELECT * FROM error_log", conn)
conn.close()

# 🧬 Choose which execution chain to trace
# Option 1: Use first error entry's execution_chain
chain_raw = df_error.iloc[0]["execution_chain"]

# Option 2: Uncomment to use a specific run_id
# target_run_id = "your-run-id"
# chain_raw = df_provider[df_provider["run_id"] == target_run_id]["execution_chain"].iloc[0]

# 🔧 Parse JSON string if needed
if isinstance(chain_raw, str):
    try:
        execution_chain = json.loads(chain_raw)
    except json.JSONDecodeError:
        execution_chain = []
else:
    execution_chain = chain_raw or []

print(f"🔗 Execution Chain Length: {len(execution_chain)}")
print("─" * 100)

# 🔁 Step 3: Walk through the execution chain
for i, run_id in enumerate(execution_chain):
    matches = df_provider[df_provider["run_id"] == run_id]
    if matches.empty:
        print(f"\n🔹 Step {i + 1}: Run ID {run_id} not found in provider_log")
        continue

    row = matches.iloc[0]
    print(f"\n🔹 Step {i + 1}")
    print(f"🧬 Run ID     : {row['run_id']}")
    print(f"📦 Provider   : {row['provider_type']} (ID {row['provider_id']})")
    print(f"📄 File Log   : {row.get('file_log_id')}")
    print(f"⏱️  Latency   : {row.get('latency_ms')} ms")
    print(f"🕒 Timestamp  : {row['timestamp']}")
    print(f"🧭 Called By  : {row.get('called_by_type')} (ID {row.get('called_by_id')})")
    print(f"🧾 Output Sch.: {row.get('output_schema')}")
    print(f"📝 Summary    :\n{row.get('output')[:500]}")  # Trim large payloads
    print("─" * 100)


🔗 Execution Chain Length: 5
────────────────────────────────────────────────────────────────────────────────────────────────────

🔹 Step 1
🧬 Run ID     : d8d7f030-a681-43df-883b-94445e0fa23b
📦 Provider   : PROVIDER_TYPE.PROGRAM (ID 1)
📄 File Log   : 1
⏱️  Latency   : 22584 ms
🕒 Timestamp  : 2025-06-17T11:07:32.623346+00:00
🧭 Called By  : PROVIDER_TYPE.SESSION (ID 1)
🧾 Output Sch.: ProgramOutputSchema
📝 Summary    :
{"state": "end", "previous_state": "preprocessing_controller", "state_type": "end", "decision": "accepted", "steps": 2, "max_steps": 20, "summary": "completed successfully", "output": {"state": "end", "file_path": "working_files\\complex_lint_task.py", "session_id": 1, "file_log_id": 1, "reason": "completed successfully", "steps": 2, "retry_count": 0, "_last_state": "preprocessing_controller", "decision": "accepted", "original_file": "tests\\complex_lint_task.py", "run_id": "d8d7f030-a681-43df-8
────────────────────────────────────────────────────────────────────────────────